# Games operations sample: build the semantic model, then check the ontology

Creates the **Games Operations Model** semantic model (Direct Lake) from the lakehouse tables with everything the
ontology needs: key columns, relationships, measures and the NOC row-level security role. All definitions come from
`generator/ontology_spec.py`, so the model always matches `ontology/ontology-design.md`.

The ontology is then created in the portal with one click (**Generate ontology**) and finished with two prompts
to the ontology agent (`ontology/ontology-agent-prompts.md`). The last section of this notebook checks the result
against the design.

**Before running**
- `01_load_lakehouse.ipynb` has run, and this notebook has `GamesLakehouse` attached as its default lakehouse.
- The workspace is not *My workspace*, and the capacity allows XMLA read/write (the default on Fabric capacities).
- Run as a workspace Contributor or higher.

Safe to re-run: existing relationships, measures and roles are left in place or updated.

In [1]:
%pip install -q "semantic-link-labs>=0.15"

^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Parameters
SEMANTIC_MODEL = "Games Operations Model"
LAKEHOUSE = "GamesLakehouse"
WORKSPACE = None            # None = this notebook's workspace
OVERWRITE_MODEL = False     # True = delete and recreate the semantic model
ONTOLOGY = "GamesOperationsOntology"

# Sign-in names of up to three test users for the row-level security test (evaluation question Q14).
# Leave a value empty to keep the placeholder for that NOC.
TEST_USERS = {"NOR": "", "SUI": "", "JPN": ""}

GEN_DIR = "/lakehouse/default/Files/games_ops/generator"
EVAL_CSV = "/lakehouse/default/Files/games_ops/output/eval/questions.csv"

In [ ]:
import sys, time, re
import pandas as pd
sys.path.insert(0, GEN_DIR)
import ontology_spec as spec
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

SM = spec.SEMANTIC_MODEL
assert not spec.validate(), spec.validate()
present = [t for t in SM["tables"] if spark.catalog.tableExists(t)]
missing = [t for t in SM["tables"] if t not in present]
print("Tables in the model:", ", ".join(present))
if missing:
    print("Not in the lakehouse (skipped):", ", ".join(missing))
    assert set(missing) <= set(SM["optional_tables"]), f"Required tables missing: {missing}"

## 1. Prepare the lakehouse

Adds the `session_event_id` key column when the data was generated before it existed, and writes the test users
into `security_noc_user`. Both tables are small, so they are rewritten in full.

In [ ]:
def rewrite(table, pdf):
    spark.createDataFrame(pdf).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table)

se = spark.table("session_event").toPandas()
if "session_event_id" not in se.columns or se["session_event_id"].isna().any():
    se["session_event_id"] = se.session_id + "|" + se.event_id
    se = se[["session_event_id"] + [c for c in se.columns if c != "session_event_id"]]
    rewrite("session_event", se)
    print("Added session_event_id")

users = spark.table("security_noc_user").toPandas()
changed = False
for noc, upn in TEST_USERS.items():
    if upn:
        users.loc[users.noc_code == noc, "user_principal_name"] = upn
        changed = True
if changed:
    rewrite("security_noc_user", users)
display(users)

## 2. Create the Direct Lake semantic model

In [ ]:
existing = fabric.list_datasets(workspace=WORKSPACE)
exists = SEMANTIC_MODEL in set(existing["Dataset Name"])
if exists and not OVERWRITE_MODEL:
    print(f"'{SEMANTIC_MODEL}' exists; keeping it and updating it in the next steps.")
    with connect_semantic_model(dataset=SEMANTIC_MODEL, readonly=True, workspace=WORKSPACE) as tom:
        in_model = {t.Name for t in tom.model.Tables}
    to_add = {t: t for t in present if t not in in_model}
    if to_add:
        with connect_semantic_model(dataset=SEMANTIC_MODEL, readonly=False, workspace=WORKSPACE) as tom:
            tom.add_direct_lake_tables(tables=to_add)
        print("Added tables:", ", ".join(to_add))
else:
    labs.directlake.generate_direct_lake_semantic_model(
        dataset=SEMANTIC_MODEL, tables=present, source=LAKEHOUSE, source_type="Lakehouse",
        workspace=WORKSPACE, refresh=False, overwrite=OVERWRITE_MODEL)
    print(f"Created '{SEMANTIC_MODEL}'")

## 3. Keys, relationships, measures and roles

Marking key columns matters for the ontology: generated relationship types are bound to data only when the key
of the "one" side is identified, and the key becomes the entity type key.

In [ ]:
rels_now = fabric.list_relationships(dataset=SEMANTIC_MODEL, workspace=WORKSPACE)
have_rel = {(r["From Table"], r["From Column"], r["To Table"], r["To Column"]) for _, r in rels_now.iterrows()}
meas_now = fabric.list_measures(dataset=SEMANTIC_MODEL, workspace=WORKSPACE)
have_meas = set(meas_now["Measure Name"]) if len(meas_now) else set()
log = []

with connect_semantic_model(dataset=SEMANTIC_MODEL, readonly=False, workspace=WORKSPACE) as tom:
    tables = {t.Name for t in tom.model.Tables}

    for t, c in SM["keys"].items():
        if t in tables:
            tom.model.Tables[t].Columns[c].IsKey = True
    log.append("keys marked")

    for ft, fc, tt, tc, state in SM["relationships"]:
        if ft not in tables or tt not in tables or (ft, fc, tt, tc) in have_rel:
            continue
        one_to_one = "one-to-one" in state
        tom.add_relationship(from_table=ft, from_column=fc, to_table=tt, to_column=tc,
                             from_cardinality="One" if one_to_one else "Many", to_cardinality="One",
                             cross_filtering_behavior="BothDirections" if one_to_one else "OneDirection",
                             is_active=state.startswith("active"))
        log.append(f"relationship {ft}[{fc}] -> {tt}[{tc}] ({state})")

    for name, table, dax, fmt, _ in SM["measures"]:
        if name in have_meas:
            m = next(m for m in tom.all_measures() if m.Name == name)
            m.Expression, m.FormatString = dax, fmt
            log.append(f"measure updated: {name}")
        else:
            tom.add_measure(table_name=table, measure_name=name, expression=dax, format_string=fmt)
            log.append(f"measure added: {table}[{name}]")

    roles = {r.Name for r in tom.model.Roles}
    for role in SM["roles"]:
        if role["name"] not in roles:
            tom.add_role(role_name=role["name"], model_permission="Read", description=role["note"])
            log.append(f"role added: {role['name']}")
        if role["filter"]:
            tom.set_rls(role_name=role["name"], table_name=role["table"], filter_expression=role["filter"])
    members = [u for u in TEST_USERS.values() if u]
    if members:
        tom.add_role_member(role_name=SM["roles"][0]["name"], member=members)
        log.append(f"role members: {', '.join(members)}")

    for t in SM["hidden_tables"]:
        if t in tables:
            tom.model.Tables[t].IsHidden = True

print("\n".join(log))

In [ ]:
labs.refresh_semantic_model(dataset=SEMANTIC_MODEL, workspace=WORKSPACE)
print("Refreshed")

## 4. Check the model against the expected answers

In [ ]:
expected = pd.read_csv(EVAL_CSV).set_index("id")["expected_answer"]

medals = fabric.evaluate_dax(SEMANTIC_MODEL, '''
EVALUATE
TOPN ( 3,
    SUMMARIZECOLUMNS ( 'noc'[noc_code], "Gold", [Gold Medals], "Silver", [Silver Medals], "Bronze", [Bronze Medals] ),
    [Gold], DESC, [Silver], DESC )
''', workspace=WORKSPACE)
display(medals)
print("Expected Q03:", expected["Q03"])

att = fabric.evaluate_dax(SEMANTIC_MODEL, '''
EVALUATE ROW ( "Attendance rate, Valmont Ice Arena", CALCULATE ( [Attendance Rate], 'venue'[venue_id] = "V01" ) )
''', workspace=WORKSPACE)
display(att)
print("Expected Q04:", expected["Q04"])

## 5. Create the ontology (portal, about 10 minutes)

1. Open **Games Operations Model** in the workspace and select **Generate ontology** (it can be under **…**).
   Name it `GamesOperationsOntology` and select **Create**. This creates the entity types, properties, data
   bindings, relationship types and metrics.
2. Open the ontology, select **Ontology agent**, attach `ontology/ontology-spec.json`, and paste **prompt 1**
   from `ontology/ontology-agent-prompts.md` (renames, clean-up, keys, relationships). Review the preview, switch
   to **Act** mode and ask it to apply.
3. Paste **prompt 2** (real-time data from the eventhouse and business rules). Review, then apply in **Act** mode.
4. Refresh the ontology, then run the next cell to check the result.

Test the role in the portal: semantic model **…** → **Security** → **Test as role**. Row-level security applies only
to users with Viewer-level access; the test users also need read access to the lakehouse.

## 6. Check the ontology against the design

In [ ]:
client = fabric.FabricRestClient()
ws_id = fabric.get_workspace_id() if WORKSPACE is None else fabric.resolve_workspace_id(WORKSPACE)

def lro(resp):
    if resp.status_code != 202:
        resp.raise_for_status()
        return resp.json()
    path = resp.headers["Location"].split("api.fabric.microsoft.com/", 1)[1]
    for _ in range(60):
        time.sleep(int(resp.headers.get("Retry-After", 5)))
        st = client.get(path).json()
        if st.get("status") in ("Succeeded", "Failed"):
            break
    assert st.get("status") == "Succeeded", st
    return client.get(path + "/result").json()

onts = client.get(f"v1/workspaces/{ws_id}/ontologies").json().get("value", [])
ont = next((o for o in onts if o["displayName"] == ONTOLOGY), None)
assert ont, f"No ontology named {ONTOLOGY} in this workspace"
import base64
parts = {p["path"]: base64.b64decode(p["payload"]).decode("utf-8", "replace")
         for p in lro(client.post(f"v1/workspaces/{ws_id}/ontologies/{ont['id']}/getDefinition"))["definition"]["parts"]}

norm = lambda x: re.sub(r"[^a-z0-9]", "", x.split("#")[-1].lower())
entities = {norm(p[len("entities/"):-len(".tmdl")]): p for p in parts if p.startswith("entities/")}
rel_text = parts.get("entityRelationships.tmdl", "")
rels = {norm(m) for m in re.findall(r"^entityRelationship\s+'?([^'\n]+)'?", rel_text, flags=re.M)}
rules = {norm(p[len("rules/"):-len(".tmdl")]) for p in parts if p.startswith("rules/")}

rows = []
for e in spec.ENTITY_TYPES:
    body = parts.get(entities.get(norm(e["name"]), ""), "")
    ts = [t["table"] for t in e.get("time_series", [])]
    rows.append(dict(kind="entity type", name=e["name"], found=norm(e["name"]) in entities,
                     note=("time series bound" if ts and all(t.lower() in body.lower() for t in ts)
                           else f"time series missing: {', '.join(ts)}" if ts else "")))
for n, s, t, *_ in spec.RELATIONSHIP_TYPES:
    rows.append(dict(kind="relationship", name=f"{s} {n} {t}", found=norm(n) in rels, note=""))
for r in spec.RULES:
    rows.append(dict(kind="rule", name=r["name"], found=norm(r["name"]) in rules, note=""))
extra = sorted(set(entities) - {norm(e["name"]) for e in spec.ENTITY_TYPES})
report = pd.DataFrame(rows)
display(report)
print(f"{report.found.sum()} of {len(report)} design items found.",
      ("Entity types not in the design: " + ", ".join(extra)) if extra else "")
if not rels:
    print("Relationship names could not be read from entityRelationships.tmdl; first lines:\n", rel_text[:600])

Items reported as not found are either missing or named differently. Ask the ontology agent to fix them, or
adjust them in the ontology editor, then re-run this cell. Once the result matches, export the definition for
source control: the same `getDefinition` call returns the TMDL parts.